Предполагается, что в рамках данного задания ученики сами будут изучать документацию, как минимум используя функционал `tab` и `shift+tab` от Jupyter

1. Импортируйте библиотеку pandas 
2. Считайте файл `wage.csv` в DataFrame с названием `wage`. Используйте для этого функцию pd.read_csv()

`person_id` - идентификатор человека

`gender` - пол, 0 - женский, 1 - мужской

`wage` - заработная плата в рублях

In [ ]:
import pandas as pd
wage = pd.read_csv('wage.csv')

[[2, 4], [6, 8], [10, 12]]


TypeError: list indices must be integers or slices, not tuple

3. Поменяйте колонку `gender` так, чтобы были записи `F` (female) и `M` (male) вместо 0 и 1

In [332]:
wage.gender = wage.gender.apply(lambda el: 'F' if el == 0 else "M")

4. Давайте посчитаем среднюю зарплату у мужчин и женщин. Для этого используйте метод `groupby` со следующим синтаксисом:

`dataframe.groupby(Название_колонки_для_группировки)[Перечисление_колонок_для_агрегации].функция_агрегации()`

In [321]:
print(wage.groupby('gender')['wage'].mean())

gender
F    40855.747261
M    46815.944005
Name: wage, dtype: float64


5. Теперь взглянем внимательнее на данные и обнаружим, что некоторые люди попали в выборку несколько раз. 
    1. Найдите таких людей. Подсказка: `value_counts()`
    0. Убедитесь, что записи по ним с одинаковым `wage`. Возможно, тут вам пригодится функция агрегации `nunique()`, отображающая количество разных значений
    0. Избавьтесь от повторяющихся значений. Подсказка: `drop_duplicates()`

In [337]:
print(wage.value_counts())
print('')

a = [group for _, group in wage.groupby('person_id') if len(group) > 1] #if (group['wage'].value_counts() != group['wage'].nunique()).any()]
print(a)
wage = wage.drop_duplicates()
a = [group for _, group in wage.groupby('person_id')]
print(a)


person_id  gender  wage         
0          M       46793.603811     1
671        F       10248.867587     1
658        F       19783.977951     1
659        F       10461.600089     1
660        M       65520.795238     1
                                   ..
338        F       24467.660296     1
339        M       19057.252451     1
340        F       69126.349194     1
341        F       25954.347353     1
999        M       108107.141368    1
Length: 1000, dtype: int64

[]
[   person_id gender          wage
0          0      M  46793.603811,    person_id gender         wage
1          1      M  33481.57572,    person_id gender          wage
2          2      M  44523.699084,    person_id gender          wage
3          3      M  15995.576829,    person_id gender          wage
4          4      F  10282.631224,    person_id gender          wage
5          5      M  65464.532281,    person_id gender          wage
6          6      M  35395.172454,    person_id gender           wage
7

6. Теперь посмотрим внимательнее на зарплаты
    1. Охарактеризуйте имеющиеся данные по зарплатам. Подсказка: `describe`
    1. Избавьтесь от бессмысленных значений

In [323]:
print(wage.describe())
wage = wage[wage.wage > 0]
print(wage.describe())


         person_id           wage
count  1000.000000    1000.000000
mean    499.500000   43694.227404
std     288.819436   55352.539343
min       0.000000 -287418.645743
25%     249.750000   14489.682367
50%     499.500000   27309.529498
75%     749.250000   52021.080258
max     999.000000  755320.874132
        person_id           wage
count  995.000000     995.000000
mean   501.844221   44306.969585
std    287.638377   54302.194392
min      0.000000     945.648458
25%    253.500000   14683.306148
50%    502.000000   27519.361794
75%    750.500000   52267.313664
max    999.000000  755320.874132


7. Давайте теперь посмотрим на зарплату с учетом бонуса. Для этого нам понадобится таблица `bonus.csv`. Считайте ее в переменную `bonus`. Заметьте, что она сохранена немного в другом формате, и вам понадобится уточнить параметр `sep` - разделитель записей. Сравните текущий файл с предыдущим и попробуйте решить проблему

In [ ]:
bonus = pd.read_csv('bonus.csv', sep = ';')

8. Чтобы посчитать итоговую зарплату, нам нужно по каждому человеку знать и оклад, и премию. Для этого надо будет соединить (сджойнить) таблицы по `person_id`. Используйте для этого функцию `pd.merge`. Помните, что параметр `how` должен быть `'outer'`, чтобы сохранить те записи, что есть только в одной таблице. Результат запишите в новый dataframe `df`

In [325]:
df = pd.merge(wage, bonus, how='left', on='person_id')
print(df)

     person_id gender           wage         bonus
0            0      M   46793.603811  3.332934e+04
1            1      M   33481.575720           NaN
2            2      M   44523.699084  3.192912e+06
3            3      M   15995.576829  2.196858e+04
4            4      F   10282.631224           NaN
..         ...    ...            ...           ...
992        995      M   66503.737185  3.452137e+03
993        996      M    9972.956272  3.892790e+05
994        997      F  104504.616392  5.380978e+04
995        998      M   98927.903076           NaN
996        999      M  108107.141368           NaN

[997 rows x 4 columns]


9. Наконец, давайте посчитаем итоговую зарплату
    1. Замените отсутствующие записи в колонке `bonus` нулями
    1. Уберите людей без `wage` - это те "плохие" записи, от которых мы избавлялись на предыдущих шагах
    1. Сделайте новую колонку `total`, которая будет равна 12 окладам и премии
    1. Посчитайте среднюю и медианную итоговую зарплату в разрезе по полу. Подсказка: вместо функции агрегации можно написать `.agg()` и перечислить внутри нужные агрегаты

In [ ]:
df = pd.merge(wage, bonus, how='left', on='person_id')
df.bonus = df.bonus.apply(lambda el: 0 if pd.isna(el) else el)
df['total'] = df.bonus + df.wage*12
df = df.groupby('gender').agg(agg'median')
print(df)

        person_id          wage  bonus          total
gender                                               
F           490.0  26537.201614    0.0  346233.018846
M           508.5  28794.605952    0.0  435581.404242


10. Сохраните `df` в файл, используя метод `to_csv()`. Не записывайте индексы

In [330]:
df.to_csv('res.csv', )